In [1]:
from nrem_sc.constants import PROCESSED_DATA_PATH

import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap

from replay_trajectory_classification import (
    SortedSpikesClassifier,
    Environment,
    RandomWalk,
    Uniform,
    Identity,
    DiagonalDiscrete,
    make_track_graph,
)

unit_id = '116b'
STATE_PROB = 0.99

def get_environment(num_nodes: int = 360, place_bin_size: float = 1.0):
    radius = 180 / np.pi
    angle = np.linspace(2 * np.pi, 0, num=num_nodes, endpoint=False)
    node_positions = np.stack((radius * np.cos(angle), radius * np.sin(angle)), axis=1)

    node_ids = np.arange(node_positions.shape[0])
    edges = np.stack((node_ids, np.roll(node_ids, shift=1)), axis=1)

    track_graph = make_track_graph(node_positions, edges)

    n_nodes = len(track_graph.nodes)
    edge_order = np.stack(
        (np.roll(np.arange(n_nodes - 1, -1, -1), 1),
         np.arange(n_nodes - 1, -1, -1)),
        axis=1,
    )

    return Environment(
        place_bin_size=place_bin_size,
        track_graph=track_graph,
        edge_order=edge_order,
        edge_spacing=0,
    )

def build_classifier(environment: Environment) -> SortedSpikesClassifier:
    continuous_transition_types = [
        [RandomWalk(movement_var=2.0), Uniform(), Identity()],
        [Uniform(),                    Uniform(), Uniform()],
        [RandomWalk(movement_var=2.0), Uniform(), Identity()],
    ]
    return SortedSpikesClassifier(
        environments=environment,
        continuous_transition_types=continuous_transition_types,
        discrete_transition_type=DiagonalDiscrete(STATE_PROB),
    )

# Load data
sleep_states    = nap.load_file(PROCESSED_DATA_PATH / unit_id / "sleep.npz")
hd_spikes       = nap.load_file(PROCESSED_DATA_PATH / unit_id / "hd_spikes_filtered.npz")
hd_angle        = nap.load_file(PROCESSED_DATA_PATH / unit_id / "angle_openfield.npz")
sessions        = nap.load_file(PROCESSED_DATA_PATH / unit_id / "sessions_labeled.npz")

In [2]:
sessions

  index     start       end  label
      0      0      1551.35  openfield
      1   1551.35  16644.5   homecage
      2  16644.5   18783.1   openfield
      3  18783.1   21454     ttx
      4  21454     22244.2   openfield
      5  22244.2   62109.4   homecage
      6  62109.4   63443.9   openfield
shape: (7, 2), time unit: sec.

In [5]:
hd_angle.time_support

  index    start      end
      0  16663.1  18772.6
shape: (1, 2), time unit: sec.

In [ ]:
# 2 ms bins
ep = nap.IntervalSet([16800], [18000])
spikes = hd_spikes.count(bin_size=2, ep=ep, time_units='ms').astype(np.bool)
t, spikes = spikes.times(), spikes.to_numpy()
angle = hd_angle.bin_average(bin_size=2, ep=ep, time_units='ms').to_numpy()

is_training = np.ones_like(angle, dtype=bool)
is_training[int(len(angle)*0.75):] = False
is_training